# Robotic Motor Rehab Cross Subject Example submission

Greetings welcome to the UKBCIC Motor rehab control cross subject challenge. This notebook contains an example of a Common Spatial Patterns + Support vector machine pipeline for predicting intended motor from the EEG. If you just want to see an example or you are looking where to start, please read through the code in the notebook below. EEG data can sometimes be hard to work with, so feel free to copy any of the functions for loading, segmenting organising data to your own solution.

This notebook is also an example of how to submit the submission.csv file in the correct order and format.

In [ ]:
import numpy as np 
import pandas as pd 

all_files = []
train_files = []
test_files = []

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        file_path = os.path.join(dirname, filename)
        all_files.append(file_path)
        if 'test' in filename:
            test_files.append(file_path)
        else:
            train_files.append(file_path)

test_files.sort() 
import kagglehub

In [ ]:
if len(os.listdir('/kaggle/input/competitions/low-cost-motor-imagery-decoding-for-rehab-cross-subject/Cross participant test set')) != len(test_files):
    raise ('Test and train file miss match! Are you sure you loaded all files?')
else: print('Loaded all test files')

if len(os.listdir('/kaggle/input/competitions/low-cost-motor-imagery-decoding-for-rehab-cross-subject/Training set')) != len(train_files):
    raise ('Test and train file miss match! Are you sure you loaded all files?')
else: print('Loaded all train files')


## Define a class or function to load our data

In [ ]:
class CSV_loader():
    freq = None
    overlap = None
    epoch_length = None

    def __init__(self, freq: int = 250, overlap: float = 0.5, epoch_length: float = 1.0):

        self.freq = freq
        self.overlap = overlap
        self.epoch_length = epoch_length

    def load_MI_CSV(self, data_dir):

        df = pd.read_csv(data_dir)

        data_columns = ['Fz','C3','Cz','C4','PO7','Pz','PO8','Oz']
        event_columns = []

        for name in df.columns:
            if "Marker" in name:
                event_columns.append(name)

        data_df = df[data_columns]
        event_df = df[event_columns]

        events = event_df["Marker_val"]
        triggers = events.index[events.notna()].tolist()
        trig_values = events.iloc[triggers].tolist()

        data = data_df.to_numpy()

        pos_dict = {'move': 1,
                   'rest': 0}

        instructions = [[],[]]


        for i in range(len(triggers) - 1):
            if trig_values[i] in ['move','rest']:
                instructions[int(pos_dict[trig_values[i]])].extend(self.cut_instance(data[triggers[i]:triggers[i + 1], ...]))
            else:
                pass
        return instructions

    def load_MI_CSV_test(self, data_dir):

        df = pd.read_csv(data_dir)

        data_columns = ['Fz','C3','Cz','C4','PO7','Pz','PO8','Oz']
        event_columns = []

        for name in df.columns:
            if "Marker" in name:
                event_columns.append(name)

        data_df = df[data_columns]
        event_df = df[event_columns]
        events = event_df["Marker_val"]
        triggers = events.index[events.notna()].tolist()
        trig_values = events.iloc[triggers].tolist()
        data = data_df.to_numpy()
        
        instructions = []

        for i in range(len(triggers) - 1):
            if trig_values[i] in ['cue_start']:
                instructions.append(self.cut_instance(data[triggers[i]:triggers[i + 1], ...]))
            else:
                pass
        return instructions
    

    def strip_string(self, s):
        return s.strip()


    def cut_instance(self, data):

        temp_list = []

        pointer = 0
        jump = int(self.freq * self.overlap)
        epoch_len = int(self.freq * self.epoch_length)

        while (pointer + epoch_len < data.shape[0]):
            temp_list.append(data[pointer:pointer + epoch_len, :])
            pointer += jump

        return temp_list

## Load our data

In [ ]:
loader = CSV_loader()
all_data = [[],[]]
for f in train_files:
    print(f'Loading {os.path.split(f)[1]}')
    temp_data = loader.load_MI_CSV(f)
    all_data[0].extend(temp_data[0])
    all_data[1].extend(temp_data[1])

all_data[0] = np.stack(all_data[0], axis = 0) #Rest examples
all_data[1] = np.stack(all_data[1], axis = 0) #Train examples
print(all_data[0].shape)
print(all_data[1].shape)

In [ ]:
y = []
for i in range(len(all_data)):
    y.append(np.zeros(all_data[i].shape[0])+i)
y = np.concatenate(y, axis = 0)
X = np.concatenate(all_data, axis = 0)

print(X.shape)
print(y.shape)
print(f'Unique y values {np.unique(y)}')

## Define our classifier 

In [ ]:
import mne
# mne.set_log_level('CRITICAL')
from mne.decoding import CSP
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

class CSP_SVM:

    def __init__(self):
        self.pipeline = Pipeline([
            ('csp', CSP()),
            ('svm', SVC(class_weight= 'balanced'))
        ])

    def fit (self,X, y):
        X = np.swapaxes(X, 1,2)
        self.pipeline.fit(X,y)
    def predict_all(self,X):
        X = np.swapaxes(X, 1,2)
        return self.pipeline.predict(X)

    def predict(self,X):
        return self.pipeline.predict(X[np.newaxis,:,:])

    def evaluate(self,X, y):
        X = np.swapaxes(X, 1,2)
        scores = cross_val_score(self.pipeline, X, y, cv = 5)
        mean = np.mean(scores)
        # print(f'within file mean accuracy {mean}')
        return mean

    def plot_csp_features(self,X, y):
        X = np.swapaxes(X,1,2)
        csp = CSP()
        csp.fit(X,y)
        info = mne.create_info(
            ch_names=['Fz','C3','Cz', 'C4','PO7','Pz','PO8' ,'Oz'], 
            sfreq=250.0,
            ch_types='eeg')
        info.set_montage('standard_1020')
        
        fig = csp.plot_filters(info, ch_type='eeg', units='a.u.')
        fig.show()

        fig = csp.plot_patterns(info, ch_type='eeg', units='a.u.')
        fig.show()

## Process our data and fit our classifier

In [ ]:
import scipy as sci
b,a = sci.signal.butter(4, [4, 25], btype = 'bandpass', fs = 512)
X = sci.signal.filtfilt(b,a,X, axis = 1)

In [ ]:
def z_score(X:np.ndarray):
    return (X-np.mean(X))/np.std(X)

In [ ]:
for i in range(X.shape[0]):
    X[i] = z_score(X[i])

In [ ]:
classifier = CSP_SVM()
print(f"5 Fold Accuracy: {classifier.evaluate(X, y)*100:.3f}%")

In [ ]:
classifier.plot_csp_features(X,y)

In [ ]:
classifier = CSP_SVM()
classifier.fit(X,y)

## Predicting on our data

In [ ]:
test_data = []
for tf in test_files:
    print(f'Loading {os.path.split(tf)[1]}')
    test_data.extend(loader.load_MI_CSV_test(tf))

print(f'{len(test_data)} test examples')

In [ ]:
# label_mapping = {1:'move',0:'rest'}
prediction_dicts = []
for idx, data in enumerate(test_data):
    predictions = []
    for epoch in data:

        b,a = sci.signal.butter(4, [4, 25], btype = 'bandpass', fs = 512)
        epoch = sci.signal.filtfilt(b,a,epoch, axis = 0)
        epoch = z_score(epoch)
        predictions.append(int(classifier.predict(epoch.T)[0]))
        
    if predictions.count(1) > predictions.count(0):
        prediction_dicts.append({'ID': idx,
                                'TARGET':'move'})
    else:
        prediction_dicts.append({'ID': idx,
                                'TARGET':'rest'})  

## Creating and submitting our csv file

In [ ]:
output_df = pd.DataFrame(prediction_dicts)
output_df.info()

In [ ]:
output_df.to_csv('submission.csv', index=False)
print("Submission created")